# 第 7 章 · 多 agent（LoopAgent）

一个干活的 agent + 一个检查状态的自定义 agent，循环到**条件满足**或撞 `max_iterations`。

⚠️ 书里那版永远停不下来：instruction 让模型"把 status 设成 completed"，
**但 LlmAgent 写不了 state**——它只能"说"完成了，状态纹丝不动。下面用工具补上。

★ 本版把"完成任务"做成**随机事件**（每轮 `DONE_PROB` 概率成功），所以能真正看到
循环跑多轮、以及"条件停"和"预算停"的区别，而不是一轮就退出。

kernel 选 `Python (agentic-adk)`。

In [1]:
import sys
import uuid
from pathlib import Path

# 【适配】定位仓库根；后端（官方 / 中转站）由 .env 的 ADK_BACKEND 决定
_root = Path.cwd().resolve()
for _d in (_root, *_root.parents):
    if (_d / "common").is_dir():
        sys.path.insert(0, str(_d))
        break
from common import adk_model, quiet_adk_sync_warnings, retry_call
quiet_adk_sync_warnings()

try:
    from google.adk.runners import InMemoryRunner
    from google.genai import types
except ModuleNotFoundError:
    # 【适配】内核选错时的提示；比裸 ModuleNotFoundError 好定位
    raise RuntimeError(
        "本 notebook 需要内核『Python (agentic-adk)』，当前内核里没有 google.adk。\n"
        "VS Code 里点右上角内核名 → 选 Python (agentic-adk) → 重新运行本格。"
    ) from None


def run_once(agent, message):
    """跑一次，返回 (每个 agent 的最终答复, 结束时 session.state)。"""
    runner = InMemoryRunner(agent)
    sid = str(uuid.uuid4())
    runner.session_service.create_session_sync(
        app_name=runner.app_name, user_id="u", session_id=sid)
    finals = []
    for ev in retry_call(lambda: list(runner.run(
            user_id="u", session_id=sid,
            new_message=types.Content(role="user", parts=[types.Part(text=message)]))), label="run "):
        if ev.is_final_response() and ev.content:
            t = getattr(ev.content, "text", None) or "".join(
                p.text for p in (ev.content.parts or []) if getattr(p, "text", None))
            finals.append((getattr(ev, "author", "?"), (t or "").strip()))
    sess = runner.session_service.get_session_sync(
        app_name=runner.app_name, user_id="u", session_id=sid)
    return finals, sess.state


import random
from typing import AsyncGenerator

from google.adk.agents import BaseAgent, LlmAgent, LoopAgent
from google.adk.agents.invocation_context import InvocationContext
from google.adk.events import Event, EventActions
from google.adk.tools import ToolContext
from google.genai import types

DONE_PROB = 0.4          # ★ 每轮"完成任务"的概率；调低能看到更多轮


def try_finish(tool_context: ToolContext) -> str:
    """尝试完成当前步骤。成功返回 DONE 并把 status 置为 completed；否则返回 NOT_DONE。"""
    # 【适配】书里让模型直接改 state —— LlmAgent 做不到，只能通过工具写
    n = tool_context.state.get("attempts", 0) + 1
    tool_context.state["attempts"] = n
    ok = random.random() < DONE_PROB
    tool_context.state.setdefault("log", []).append(
        f"try_finish#{n} -> {'DONE' if ok else 'NOT_DONE'}")
    if ok:
        tool_context.state["status"] = "completed"
        return "DONE"
    return "NOT_DONE: 本轮没做成，还有问题没解决"


class ConditionChecker(BaseAgent):
    """读 state 里的 status 决定要不要停。"""
    name: str = "ConditionChecker"
    description: str = "Checks if a process is complete and signals the loop to stop."

    async def _run_async_impl(self, context: InvocationContext) -> AsyncGenerator[Event, None]:
        status = context.session.state.get("status", "pending")
        if status == "completed":
            # ★ 本章新增 escalate：置为 True 就是"跳出循环"的信号
            #   （第 4 章那个 exit_loop 工具，内部做的就是这件事）
            yield Event(author=self.name, actions=EventActions(escalate=True))
        else:
            yield Event(author=self.name,
                        content=types.Content(role="model", parts=[types.Part(text="继续下一轮")]))


worker = LlmAgent(
    name="Worker",
    model=adk_model(),                      # 【适配】gemini-2.0-flash-exp 已下线
    instruction="每一轮都调用 try_finish 工具尝试完成任务。"
                "它返回 DONE 就说“已完成”，否则说“还需要继续”。",
    tools=[try_finish],
)

poller = LoopAgent(name="Poller", max_iterations=6,
                   sub_agents=[worker, ConditionChecker()])


def run_loop(seed):
    """跑一次循环。按 ConditionChecker 的执行次数计轮数（每个 Worker 回合会产生多个事件）。"""
    random.seed(seed)                       # 固定随机性，方便复现
    runner = InMemoryRunner(poller)
    sid = str(uuid.uuid4())
    runner.session_service.create_session_sync(
        app_name=runner.app_name, user_id="u", session_id=sid)
    rounds = 0
    for ev in runner.run(user_id="u", session_id=sid,
                         new_message=types.Content(role="user", parts=[types.Part(text="完成这个任务")])):
        if getattr(ev, "author", "") != "ConditionChecker":
            continue
        rounds += 1
        s = runner.session_service.get_session_sync(
            app_name=runner.app_name, user_id="u", session_id=sid).state
        last = (s.get("log") or ["(还没尝试)"])[-1]
        status = s.get("status", "pending")
        print(f"  第 {rounds} 轮: {last:<26s} status={status:<10s} "
              f"{'escalate → 退出循环' if status == 'completed' else '继续'}")
    return rounds, s.get("status")


for seed in (1, 2, 3):
    print(f"\n===== seed={seed} =====")
    rounds, status = run_loop(seed)
    print(f"  >> {rounds} 轮 | 最终 status={status!r}")


===== seed=1 =====


  第 1 轮: try_finish#1 -> DONE       status=completed  escalate → 退出循环
  >> 1 轮 | 最终 status='completed'

===== seed=2 =====


  第 1 轮: try_finish#1 -> NOT_DONE   status=pending    继续


  第 2 轮: try_finish#2 -> NOT_DONE   status=pending    继续


  第 3 轮: try_finish#3 -> DONE       status=completed  escalate → 退出循环
  >> 3 轮 | 最终 status='completed'

===== seed=3 =====


  第 1 轮: try_finish#1 -> DONE       status=completed  escalate → 退出循环
  >> 1 轮 | 最终 status='completed'
